In [1]:
import os
import sys
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

dir = "/Users/julianguyen/Documents/multimodal-pmlb"
os.chdir(dir)
sys.path.insert(0, dir)
from workflow.notebooks.utils.early_helpers import *

np.random.seed(101)

In [2]:
# load in files
atc = pd.read_csv("data/procdata/model_cg2/atc.csv", index_col=0)
rna = pd.read_csv("data/procdata/model_cg2/rna.csv", index_col=0)
cnv = pd.read_csv("data/procdata/model_cg2/cnv.csv", index_col=0)
mut = pd.read_csv("data/procdata/model_cg2/mut.csv", index_col=0)
pheno = pd.read_csv("data/procdata/model_cg2/meta.csv", index_col=0)

# get targest
dt = pheno["doubling_rate"]

# Unimodal Models without Cancer Type

In [3]:
# LASSO
#run_lasso(atc, dt, "4-CommonGene2/atac_")
#run_lasso(rna, dt, "4-CommonGene2/rna_")
#run_lasso(cnv, dt, "4-CommonGene2/cnv_")
#run_lasso(mut, dt, "4-CommonGene2/mut_")

# elastic net
a_pred = run_elastic_net(atc, dt, "4-CommonGene2/atac_", preds=True)
r_pred = run_elastic_net(rna, dt, "4-CommonGene2/rna_", preds=True)
c_pred = run_elastic_net(cnv, dt, "4-CommonGene2/cnv_", preds=True)
m_pred = run_elastic_net(mut, dt, "4-CommonGene2/mut_", preds=True)

# random forest
#run_random_forest(atc, dt, "4-CommonGene2/atac_")
#run_random_forest(rna, dt, "4-CommonGene2/rna_")
#run_random_forest(cnv, dt, "4-CommonGene2/cnv_")
#run_random_forest(mut, dt, "4-CommonGene2/mut_")

ValueError: 
All the 180 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
180 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/ml/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/opt/anaconda3/envs/ml/lib/python3.12/site-packages/sklearn/base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ml/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py", line 1025, in fit
    X, y = validate_data(
           ^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/ml/lib/python3.12/site-packages/sklearn/utils/validation.py", 

# Late Fusion Models without Cancer Type

In [ ]:
# Combine predictions into one dataframe
pred_df = pd.DataFrame({
    'ATAC': pred_atac,
    'RNA': pred_rna,
    'CNV': pred_cnv,
    'Mutation': pred_mut
})

# Simple average
pred_df['late_fusion'] = pred_df.mean(axis=1)

# Evaluate correlation with true labels
s_corr = spearmanr(dt, pred_df['late_fusion']).correlation
p_corr = pearsonr(dt, pred_df['late_fusion'])[0]

print("Late Fusion Spearman:", s_corr)
print("Late Fusion Pearson:", p_corr)